# GOV-01 Controlled Experiment: Frozen MobileNetV2

This notebook compares transfer learning with the class-weighted CNN. MobileNetV2 already learned general visual patterns from ImageNet images; here, its existing layers are frozen and only a small final classification head learns from the pothole data.

The clean split, seed, image size, training-only augmentation, class weights, validation metric, and protected-test boundary remain the same.

**Do not load or evaluate the protected test split.**

## Before running in Colab

Use a Colab runtime with the latest repository and the prepared clean split. If this is a new runtime, clone the repository, upload/extract `Dataset.zip`, and run `src/build_clean_split.py` with seed `42`.

The first run may download ImageNet MobileNetV2 weights. This is expected. If it fails because Colab cannot download the weights, stop and show the error instead of changing the model.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score, precision_score, recall_score, roc_auc_score

SEED = 42
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_NAMES = ['Normal', 'Pothole']
DATA_DIR = Path('data/processed/clean_split')
REPORTS_DIR = Path('reports')
RUN_NAME = 'mobilenetv2_frozen_v3'

tf.keras.utils.set_random_seed(SEED)
REPORTS_DIR.mkdir(exist_ok=True)

for split_name in ['train', 'validation', 'test']:
    if not (DATA_DIR / split_name).is_dir():
        raise FileNotFoundError(f'Missing folder: {DATA_DIR / split_name}')

print('Clean split found:', DATA_DIR.resolve())
print('Protected test folder exists but will not be loaded.')

In [ ]:
def load_split(split_name, shuffle):
    return tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / split_name, labels='inferred', label_mode='binary',
        class_names=CLASS_NAMES, color_mode='rgb', image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE, shuffle=shuffle, seed=SEED if shuffle else None,
    )

train_ds = load_split('train', shuffle=True)
validation_ds = load_split('validation', shuffle=False)

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.10),
], name='training_augmentation')

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda images, labels: (data_augmentation(images, training=True), labels), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
validation_ds = validation_ds.prefetch(AUTOTUNE)

print('Training and validation datasets are ready.')

In [ ]:
# Keep the same class weights as cnn_class_weighted_v2.
train_counts = {0: 236, 1: 624}
train_total = sum(train_counts.values())
class_weight = {label: train_total / (2 * count) for label, count in train_counts.items()}

print('Class weights:', class_weight)
print('Only the model family and its required ImageNet preprocessing change in this experiment.')

In [ ]:
# Frozen transfer-learning model.
tf.keras.utils.set_random_seed(SEED)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMAGE_SIZE + (3,), include_top=False, weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=IMAGE_SIZE + (3,))
x = data_augmentation(inputs, training=True)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.30)(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid', name='pothole_probability')(x)
transfer_model = tf.keras.Model(inputs, outputs, name=RUN_NAME)

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='roc_auc')],
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=4, restore_best_weights=True,
)
transfer_model.summary()

In [ ]:
start_time = time.time()
history = transfer_model.fit(
    train_ds, validation_data=validation_ds, epochs=20,
    class_weight=class_weight, callbacks=[early_stopping], verbose=1,
)
training_seconds = time.time() - start_time
print(f'Training time: {training_seconds:.1f} seconds')

In [ ]:
# Validation evaluation only. The protected test split is not loaded.
y_validation = np.concatenate([labels.numpy().ravel() for _, labels in validation_ds])
scores = transfer_model.predict(validation_ds).ravel()
predictions = (scores >= 0.50).astype(int)

transfer_metrics = {
    'accuracy': float(accuracy_score(y_validation, predictions)),
    'macro_f1': float(f1_score(y_validation, predictions, average='macro', zero_division=0)),
    'pothole_precision': float(precision_score(y_validation, predictions, pos_label=1, zero_division=0)),
    'pothole_recall': float(recall_score(y_validation, predictions, pos_label=1, zero_division=0)),
    'normal_recall': float(recall_score(y_validation, predictions, pos_label=0, zero_division=0)),
    'roc_auc': float(roc_auc_score(y_validation, scores)),
}

new_record = {
    'run_name': RUN_NAME,
    'hypothesis': 'Frozen ImageNet MobileNetV2 improves validation Macro F1 over the class-weighted CNN.',
    'changed_factor': 'model family: compact CNN to frozen MobileNetV2',
    'class_weight': f'Normal={class_weight[0]:.3f}; Pothole={class_weight[1]:.3f}',
    'training_seconds': round(training_seconds, 1),
    **transfer_metrics,
}

record_path = REPORTS_DIR / 'experiment_record.csv'
previous_records = pd.read_csv(record_path) if record_path.exists() else pd.DataFrame()
previous_records = previous_records[previous_records['run_name'] != RUN_NAME] if not previous_records.empty else previous_records
comparison = pd.concat([previous_records, pd.DataFrame([new_record])], ignore_index=True)
comparison.to_csv(record_path, index=False)
display(comparison)

print(classification_report(y_validation, predictions, target_names=CLASS_NAMES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_validation, predictions, display_labels=CLASS_NAMES)
plt.title('Validation confusion matrix: mobilenetv2_frozen_v3')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'mobilenetv2_frozen_v3_validation_confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
history_frame = pd.DataFrame(history.history)
history_frame[['loss', 'val_loss']].plot(title='Frozen MobileNetV2 training and validation loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.tight_layout()
plt.savefig(REPORTS_DIR / 'mobilenetv2_frozen_v3_learning_curve.png', dpi=150)
plt.show()

print('Saved new validation evidence in reports/.')
print('Do not use the test split until a final candidate is selected.')